# OSSEO — إعادة تدريب كاشف الكسور
اختار Runtime → Change runtime type → GPU. ارفع ملف osseo-training.zip في الخلية الأولى.
هذا تدريب جديد وليس نموذجًا مضمون الدقة. لا يفعّل النموذج على الموقع تلقائيًا.
المصدر: FracAtlas CC BY 4.0. النموذج وكود Ultralytics: AGPL-3.0.
لا ترفع صور مرضى؛ هذه الحزمة تحتوي كودًا وتقسيم بيانات عامة فقط.


In [ ]:
from google.colab import files
from pathlib import Path
import os, zipfile
uploaded=files.upload()
archive=Path('osseo-training.zip')
assert archive.exists(), 'Upload osseo-training.zip'
root=Path('/content/osseo-training');root.mkdir(exist_ok=True)
with zipfile.ZipFile(archive) as z:
    assert all((root/n.filename).resolve().is_relative_to(root.resolve()) for n in z.infolist())
    z.extractall(root)
os.chdir(root)


In [ ]:
import subprocess,sys,torch
assert torch.cuda.is_available(), 'Enable a GPU runtime first; GPU availability depends on your Colab account.'
subprocess.run([sys.executable,'-m','pip','install','ultralytics==8.4.191'],check=True)
subprocess.run([sys.executable,'ml/download.py'],check=True)


In [ ]:
# Fine-tune all layers; preserve the fixed validation/test split.
subprocess.run([sys.executable,'-u','ml/train_full.py','--device','0','--epochs','80','--batch','8'],check=True)


In [ ]:
subprocess.run([sys.executable,'-u','ml/evaluate_full.py'],check=True)
import json
report=json.loads(Path('ml/artifacts/candidate/metrics.json').read_text())
print(json.dumps(report,indent=2))
print('Meeting an internal engineering target does NOT establish clinical validity.')


In [ ]:
import zipfile
with zipfile.ZipFile('/content/osseo-candidate.zip','w',zipfile.ZIP_DEFLATED) as z:
    for name in ['best.pt','metrics.json','training.json','predictions.json']:
        p=Path('ml/artifacts/candidate')/name
        if p.exists():z.write(p,'candidate/'+name)
    z.write('ml/artifacts/full-split.json','full-split.json')
files.download('/content/osseo-candidate.zip')
